In [1]:
import gpxpy
import matplotlib.pyplot as plt
import contextily as ctx


def save_gpx_image(gpx_file_path, output_png="yatsugatake_map.png"):
    # 1. GPXデータの読み込み
    with open(gpx_file_path, 'r', encoding='utf-8') as f:
        gpx = gpxpy.parse(f)

    lats, lons = [], []
    for track in gpx.tracks:
        for segment in track.segments:
            for p in segment.points:
                lats.append(p.latitude)
                lons.append(p.longitude)

    if not lats:
        print("座標データが見つかりませんでした。")
        return

    # 2. 描画の準備
    fig, ax = plt.subplots(figsize=(12, 12))

    # 3. 緯度経度をWebメルカトールに変換してプロット
    # contextilyの関数を使って、データの範囲から背景地図を取得
    west, south, east, north = min(lons), min(lats), max(lons), max(lats)

    # 少し余裕を持たせる（マージン）
    margin = 0.01
    west, south, east, north = west - margin, south - margin, east + margin, north + margin

    # 4. 背景地図の追加
    # sourceを地理院地図（地理院タイル）にすると、日本の山岳地帯は見やすくなります
    try:
        # 緯度経度(EPSG:4326)で範囲を指定して背景を取得
        img, extent = ctx.bounds2img(west, south, east, north, ll=True, source=ctx.providers.OpenStreetMap.Mapnik)
        ax.imshow(img, extent=extent)
    except Exception as e:
        print(f"地図の取得に失敗しました: {e}")
        # 失敗した場合はOSMの標準タイルを試す
        ctx.add_basemap(ax, crs="EPSG:4326")

    # 5. 軌跡のプロット
    # ax.plot の際に、座標系を変換して描画する
    # contextilyで取得した背景の座標系に合わせて変換
    from pyproj import Transformer
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
    x, y = transformer.transform(lons, lats)

    ax.plot(x, y, color='red', linewidth=3, alpha=0.8, label='Track')

    # 6. 仕上げ
    ax.set_axis_off()
    plt.title(f"GPX Track", fontsize=15)

    # 保存
    plt.savefig(output_png, bbox_inches='tight', dpi=300)
    plt.show()  # 確認用
    print(f"Saved: {output_png}")


# 実行
save_gpx_image("data/yamap_2025-08-19_05_34.gpx")

ModuleNotFoundError: No module named 'gpxpy'